<a href="https://colab.research.google.com/github/caramos84/QC_Video/blob/main/notebooks/04_DecisionEngine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QC Video - Notebook 04
## QC Decision Engine

Este notebook ya NO consolida datos a mano: es un wrapper delgado sobre el
paquete Python `rules/` (el Motor de QC real). Toda la lógica de reglas,
scoring y generación de reporte vive en `rules/*.py` y está cubierta por
`pytest` — ver `tests/`.

Flujo:

1. Clonar el repo (para tener acceso a `rules/`, `profiles/`, `samples/`).
2. Cargar `asset_knowledge.json` (el que genera el pipeline de Notebooks 01-03).
3. Elegir un `profile` (canal/placement, ej. `meta_instagram_reels`) y,
   opcionalmente, un `brief` de campaña.
4. Ejecutar `rules.engine.run_qc(...)`.
5. Generar y descargar `qc_report.json`, `qc_score.json`, `qc_summary.md`.

Entradas:

- `asset_knowledge.json` (o un `.zip` que lo contenga)
- `profile_id` (ver `profiles/`)
- opcional: un `brief.yaml` de campaña (ver `samples/briefs/`)

Salidas:

- `outputs/<asset_id>/qc_report.json`
- `outputs/<asset_id>/qc_score.json`
- `outputs/<asset_id>/qc_summary.md`

## 1. Clonar el repo e instalar el paquete `rules/`

In [ ]:
import os

REPO_URL = "https://github.com/caramos84/QC_Video.git"
REPO_DIR = "/content/QC_Video"

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!pip install -q -e .

## 2. Cargar `asset_knowledge.json`

Acepta el archivo `asset_knowledge.json` directo, o un `.zip` que lo contenga
(por ejemplo el `asset_knowledge.zip` que descargaba la versión anterior de
este notebook).

In [ ]:
import json
import shutil
import zipfile
from pathlib import Path

from google.colab import files

uploaded = files.upload()
uploaded_name = next(iter(uploaded.keys()))

WORKDIR = Path("/content/qc_run")
if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
WORKDIR.mkdir(parents=True)

if uploaded_name.lower().endswith(".zip"):
    with zipfile.ZipFile(uploaded_name, "r") as z:
        z.extractall(WORKDIR)
    candidates = list(WORKDIR.rglob("asset_knowledge.json"))
    if not candidates:
        raise FileNotFoundError("El zip no contiene asset_knowledge.json")
    ASSET_KNOWLEDGE_PATH = candidates[0]
else:
    ASSET_KNOWLEDGE_PATH = WORKDIR / "asset_knowledge.json"
    shutil.move(uploaded_name, ASSET_KNOWLEDGE_PATH)

print(f"asset_knowledge.json listo en: {ASSET_KNOWLEDGE_PATH}")

## 3. Elegir profile (canal/placement) y brief opcional

In [ ]:
from pathlib import Path

PROFILES_DIR = Path(REPO_DIR) / "profiles"
print("Profiles disponibles:")
for p in sorted(PROFILES_DIR.glob("*.yaml")):
    print(" -", p.stem)

In [ ]:
# Editar antes de correr:
PROFILE_ID = "meta_instagram_reels"
BRIEF_PATH = Path(REPO_DIR) / "samples" / "briefs" / "example_campaign_brief.yaml"  # o None

PROFILE_PATH = PROFILES_DIR / f"{PROFILE_ID}.yaml"
assert PROFILE_PATH.exists(), f"No existe el profile: {PROFILE_PATH}"
print("Profile:", PROFILE_PATH)
print("Brief:", BRIEF_PATH if BRIEF_PATH else "(ninguno)")

## 4. Ejecutar el Motor de QC

In [ ]:
from rules.engine import run_qc_from_paths

report = run_qc_from_paths(
    ASSET_KNOWLEDGE_PATH,
    PROFILE_PATH,
    BRIEF_PATH,
)

print(f"Veredicto: {report.score_summary.verdict.value}")
print(f"Score: {report.score_summary.score}/100")
print(f"Cobertura: {report.score_summary.coverage_pct}%")

## 5. Generar artefactos de salida

`qc_report.json` (findings detallados), `qc_score.json` (score/veredicto) y
`qc_summary.md` (resumen legible, con la sección de cobertura y brechas).

In [ ]:
from rules.report import write_report

asset_id_safe = report.asset_id.replace("/", "_")
output_dir = Path("/content/qc_run/outputs") / asset_id_safe
paths = write_report(report, output_dir)

for name, path in paths.items():
    print(f"{name}: {path}")

## 6. Mostrar el resumen

In [ ]:
from IPython.display import Markdown, display

display(Markdown(paths["qc_summary"].read_text(encoding="utf-8")))

## 7. Descargar los artefactos generados

In [ ]:
import shutil as _shutil
from google.colab import files as _files

zip_base = f"/content/qc_output_{asset_id_safe}"
_shutil.make_archive(zip_base, "zip", output_dir)
_files.download(f"{zip_base}.zip")

## Resultado final

Este notebook ya no genera `asset_knowledge.json` (eso lo sigue haciendo la
consolidación previa) — genera el veredicto real de QC:

```text
outputs/<asset_id>/
├── qc_report.json
├── qc_score.json
└── qc_summary.md
```

Toda la lógica (reglas, severidades, scoring, NOT_EVALUATED por datos
faltantes) vive en `rules/*.py` y está testeada con `pytest tests/` — este
notebook es solo la interfaz interactiva sobre esa lógica.